# FoodLensVN — Kaggle notebook template

Run this notebook on a Kaggle GPU session (T4 / P100 / 16 GB).

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On** (needed for `pip install` and HuggingFace Hub).
3. **Add-ons → Secrets → add `HF_TOKEN`** with at least *read* access (write if you'll push checkpoints back). Get a token at https://huggingface.co/settings/tokens.

Dataset is pulled from HuggingFace Hub: [`Tamir39/foodlensvn`](https://huggingface.co/datasets/Tamir39/foodlensvn).

In [ ]:
# Cell 1: Clone the develop branch (or pull latest)
import os
%cd /kaggle/working/
REPO_URL = 'https://github.com/tamir39/vqa-viet-project.git'
REPO_DIR = 'vqa-viet-project'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch develop {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# Cell 2: Install deps via uv (faster + matches uv.lock)
!pip install -q uv
!uv sync --frozen 2>&1 | tail -10

In [ ]:
# Cell 3: Authenticate with HuggingFace via the Kaggle secret.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4: Path + env setup.
# Keep dataset + processed splits *inside* the repo dir so the relative paths
# in configs/base_config.yaml (data/foodlensvn, data/processed) just work.
import os, sys
ROOT = '/kaggle/working/vqa-viet-project'
os.environ['FOODLENS_DATA_DIR'] = f'{ROOT}/data/foodlensvn'
# os.environ['KAGGLE_NO_INTERNET'] = '1'  # enable when HF *model* snapshots are pre-staged

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5: GPU + library sanity check
!uv run python scripts/check_env.py

In [ ]:
# Cell 6: Pull the dataset from HF Hub (one-shot, ~34 MB).
!uv run python scripts/fetch_dataset.py --dest $FOODLENS_DATA_DIR

In [ ]:
# Cell 7: Build processed splits + answer vocab.
# Writes to ./data/processed (repo-relative) so configs/base_config.yaml finds it.
!uv run python scripts/build_dataset.py \
  --data-dir $FOODLENS_DATA_DIR \
  --output-dir data/processed \
  --image-variant squared

In [ ]:
# Cell 8: Train both modular configs back-to-back (A1 = LSTM, A2 = Transformer).
# Outputs land at reports/<config>/checkpoints/{best,last}.pt + history.json.
CONFIGS = ['configs/A1.yaml', 'configs/A2.yaml']
for cfg in CONFIGS:
    print(f'\n========== training {cfg} ==========')
    !uv run python scripts/train.py --config {cfg}

In [ ]:
# Cell 9: Push each trained checkpoint to its own HF model repo
# (Tamir39/foodlensvn-A1, Tamir39/foodlensvn-A2) so artifacts survive the session.
from huggingface_hub import HfApi, create_repo
from pathlib import Path

api = HfApi()
for cfg in CONFIGS:
    name = Path(cfg).stem                       # 'A1' / 'A2'
    repo = f'Tamir39/foodlensvn-{name}'
    local = f'reports/{name}'
    if not Path(local).is_dir():
        print(f'skip {name}: no run output at {local}')
        continue
    create_repo(repo, repo_type='model', exist_ok=True, private=False)
    api.upload_folder(
        folder_path=local,
        repo_id=repo,
        repo_type='model',
        commit_message=f'training run from Kaggle ({name})',
        ignore_patterns=['**/__pycache__/**', '*.tmp'],
    )
    print(f'{name}: pushed -> https://huggingface.co/{repo}')